# 05 — Final held-out evaluation: pure GATv2

This notebook evaluates the graph-only GATv2 model.

The trained prediction uses **no CLS logits**:

`final logits = GATv2 graph logits`

For analysis only, frozen CLS and mean-patch baselines are also computed on the exact same test episodes. They are reported separately and are never added to the GATv2 prediction.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repository:", REPO_DIR)
print("Source:", SRC_DIR)


In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)


In [ ]:
CONFIG_PATH = Path("configs/gatv2_5shot.json")
config = json.loads(CONFIG_PATH.read_text())

TEST_NUM_EPISODES = 600
TEST_SEED = 20_000
MAX_CACHED_SHARDS = 6

print(json.dumps(config, indent=2))


## Restore the test split and create fixed test episodes


In [ ]:
from cross_image_glot.storage import (
    restore_feature_splits,
    atomic_json_save,
)
from cross_image_glot.data import (
    MiniImageNetFeatureDataset,
    FewShotFeatureEpisodeDataset,
)
from cross_image_glot.baselines import evaluate_frozen_baseline
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder

restore_feature_splits(
    ["test"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

test_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir,
    "test",
    max_cached_shards=MAX_CACHED_SHARDS,
)

test_episodes = FewShotFeatureEpisodeDataset(
    test_features,
    n_way=config["n_way"],
    k_shot=config["k_shot"],
    queries_per_class=config["eval_queries_per_class"],
    num_episodes=TEST_NUM_EPISODES,
    seed=TEST_SEED,
    vary_by_epoch=False,
)

print("Test episodes:", len(test_episodes))
print(
    f"Protocol: {config['n_way']}-way "
    f"{config['k_shot']}-shot, "
    f"{config['eval_queries_per_class']} queries/class"
)


## Frozen reference baselines

These are comparison metrics only. Their logits are not supplied to GATv2.


In [ ]:
cls_metrics = evaluate_frozen_baseline(
    test_episodes,
    "cls",
    device,
    TEST_NUM_EPISODES,
    temperature=config.get("cls_temperature", 0.1),
)

mean_metrics = evaluate_frozen_baseline(
    test_episodes,
    "mean_patch",
    device,
    TEST_NUM_EPISODES,
    temperature=config["graph_temperature"],
)

print("CLS baseline:", cls_metrics)
print("Mean-patch baseline:", mean_metrics)


## Reconstruct and load the pure GATv2 model


In [ ]:
from cross_image_glot.models import (
    PatchGATv2Encoder,
    MeanPrototypeCosineReadout,
    CrossImageGraphMatcher,
)
from cross_image_glot.training import evaluate_episode_dataset

graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(test_features.metadata["grid_size"]),
    top_k=config["top_k"],
    min_similarity=None,
    graph_dtype=torch.float32,
    similarity_device=device,
)

encoder = PatchGATv2Encoder(
    input_dim=config["input_dim"],
    hidden_dim=config["hidden_dim"],
    num_layers=config["num_layers"],
    heads=config["attention_heads"],
    edge_dim=config["edge_dim"],
    dropout=config["dropout"],
)

readout = MeanPrototypeCosineReadout(
    temperature=config["graph_temperature"],
    learnable_temperature=False,
)

model = CrossImageGraphMatcher(
    encoder=encoder,
    readout=readout,
)

checkpoint_path = (
    paths.drive_checkpoint_dir
    / config["experiment_name"]
    / "best.pt"
)

if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Best pure-GATv2 checkpoint does not exist: {checkpoint_path}"
    )

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Checkpoint:", checkpoint_path)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print(
    "Best validation accuracy:",
    checkpoint.get("best_validation_accuracy", "unknown"),
)


## Final graph-only test evaluation

`evaluate_episode_dataset` receives the `CrossImageGraphMatcher` directly, so the prediction is based solely on GATv2 graph logits.


In [ ]:
model_metrics = evaluate_episode_dataset(
    model,
    graph_builder,
    test_episodes,
    device,
    TEST_NUM_EPISODES,
    config["graph_microbatch_size"],
    log_interval=20,
    split_name="test",
)

results = {
    "model_kind": "gatv2_only",
    "experiment_name": config["experiment_name"],
    "protocol": {
        "n_way": config["n_way"],
        "k_shot": config["k_shot"],
        "queries_per_class": config["eval_queries_per_class"],
        "num_episodes": TEST_NUM_EPISODES,
        "seed": TEST_SEED,
    },
    "gatv2_only": model_metrics.to_dict(),
    "cls_baseline": cls_metrics.to_dict(),
    "mean_patch_baseline": mean_metrics.to_dict(),
}

output = (
    paths.drive_results_dir
    / config["experiment_name"]
    / "test_metrics.json"
)

atomic_json_save(results, output)

print("Pure GATv2:", model_metrics)
print("CLS baseline:", cls_metrics)
print("Mean-patch baseline:", mean_metrics)
print("Saved:", output)
